<a href="https://colab.research.google.com/github/DataWithHamza/Flyrank-ML-Internship/blob/main/work/notebooks/w02_ml_task_framing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/DataWithHamza/Flyrank-ML-Internship/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task

My lane (Refresh / Content Opportunity Scoring) is best framed as a
combination of classification and scoring/ranking:

- Classification: predict whether a page falls into the "declining"
  bucket (trend_direction == "down") using its observed 90-day signals.
- Scoring/ranking: convert that classifier's probability output into a
  ranked list, so a reviewer with limited capacity works through the
  most likely candidates first (Precision@K-style usage).

I'm not choosing clustering, because I have a clear label to predict
(declining vs not), not an unlabeled grouping problem.

In [2]:
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/flyrank-bih/flyrank-ml-internship-starter"
REPO_DIR = "flyrank-ml-internship-starter"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
else:
    # find the repo root from wherever this kernel started
    while not os.path.isdir("data/raw") and os.getcwd() != "/":
        os.chdir("..")

print("Working dir:", os.getcwd())
assert os.path.exists("data/raw/content_refresh_anonymized.csv"), "starter CSV not found — are you at the repo root?"
print("Starter data found. You're ready.")

Working dir: /content/flyrank-ml-internship-starter
Starter data found. You're ready.


## 2. Target or proxy

Target: is_declining_label = (trend_direction == "down")

This label comes from a defined rule applied to the current observation
window, not a true future outcome -- so it is a proxy label, not a
directly observed future event. The lane guide flags this exact
weakness: it's a fine starting point, but a stronger version (which I
may move toward later) would use a real forward-looking label, e.g.:

  features from the prior 90 days -> decline over the NEXT 30 days

For now, at the framing stage, I'm using the starter's current-window
proxy because it's simple, transparent, and lets me validate the whole
pipeline before adding time-window complexity.

In [5]:
!{sys.executable} scripts/run_all.py


▶ Step 1/5 — Prepare features — clean the data, build the feature vector, define the label
Prepared 30,000 rows from 30,000 raw rows
Wrote /content/flyrank-ml-internship-starter/data/processed/refresh_feature_vector.csv

▶ Step 2/5 — Baseline — a transparent hand-written rule to beat
Wrote baseline queue: /content/flyrank-ml-internship-starter/data/processed/baseline_refresh_queue.csv
Top-50 declining rate (full data, not the evaluated holdout Precision@50): 0.340

▶ Step 3/5 — Train — logistic regression, decision tree, random forest (client-holdout split)
Trained 3 models on 30,000 rows
Split strategy: client_holdout
Best model: random_forest
Wrote predictions: /content/flyrank-ml-internship-starter/data/processed/model_predictions.csv
Wrote model results: /content/flyrank-ml-internship-starter/outputs/model_results.json

▶ Step 4/5 — Evaluate — ranked refresh queue, charts, and the Markdown report
Wrote final refresh queue: /content/flyrank-ml-internship-starter/outputs/refresh_que

## 3. Success metric

Primary metric: Precision@50.

Why: the content team has limited review capacity -- they can only act
on a fixed number of pages per cycle, not the whole inventory.
Precision@50 answers exactly the question that matters to them: "of the
top 50 pages the model tells us to review first, how many are actually
declining?" This matches how the ranked list will really be used, unlike
a generic accuracy score, which would be dominated by the majority class
and wouldn't reflect real-world reviewer capacity.

I'll also track ROC AUC as a secondary check of overall discriminative
power, but Precision@50 is the number I'd defend to a non-technical
stakeholder.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. The unit of analysis, as a real dataframe

One row in this dataframe = one content page (identified by content_id),
scoped to a single client and a single 90-day observation window. This
is the unit my classifier will score and my ranked queue will list.

In [6]:
import pandas as pd

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
print(f"One row = one content page.")
print(f"Shape: {df.shape[0]} rows, {df.shape[1]} columns")
df[["content_id", "client_id", "impressions_90d", "sessions_90d",
    "trend_direction", "content_age_days"]].head(5)

One row = one content page.
Shape: 30000 rows, 44 columns


,content_id,client_id,impressions_90d,sessions_90d,trend_direction,content_age_days
0,content_304f48230142,client_f369cb89fc,3803,17,down,187
1,content_a1fb4e703a9e,client_4e07408562,15320,9,down,445
2,content_9aa793d4d895,client_7f2253d7e2,12581,11,down,141
3,content_331d6c4de07b,client_19581e27de,11751,78,stable,463
4,content_d99b7a2d90ca,client_3fdba35f04,19140,145,down,263


## 5. Why ML beats a fixed rule here

A fixed if-statement rule (like the baseline: 0.40*visibility +
0.30*freshness_risk + 0.25*position_opportunity + 0.05*depth_gap) can
only combine signals in one fixed, linear way that a human guessed in
advance. Real pages don't decline for one clean reason -- a page might
be declining because of a mix of stale content AND weak CTR AND a
mid-pack position, where no single threshold captures the interaction
between these signals.

The numbers back this up: the hand-written rule scores 0.240 on
Precision@50, while the random forest scores 0.740 -- roughly a 3x lift.
A model can learn these messy, non-linear combinations of signals
automatically instead of a human having to guess and hand-tune every
threshold and weight.

In [7]:
import json
res = json.load(open("outputs/model_results.json"))

base_p50 = res["baseline"]["baseline_precision_at_50"]
rf_p50   = res["models"]["random_forest"]["precision_at_50"]

print(f"Hand-written rule Precision@50: {base_p50:.3f}")
print(f"Random forest Precision@50:     {rf_p50:.3f}")
print(f"Lift: {rf_p50/base_p50:.1f}x")

Hand-written rule Precision@50: 0.240
Random forest Precision@50:     0.740
Lift: 3.1x


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.